In [0]:
# ============================================================
# TOPIC 31 — CELL TOWER HEALTH & DROPPED CALLS
# SILVER LAYER
# ============================================================

MY_ID = "tanmaya"

CATALOG = "workspace"
SCHEMA = f"capstone_{MY_ID}"

BRONZE_CALLS = f"{CATALOG}.{SCHEMA}.bronze_calls"

SILVER_CALLS = f"{CATALOG}.{SCHEMA}.silver_calls"
SILVER_REJECTS = f"{CATALOG}.{SCHEMA}.silver_rejects"

print("Bronze calls :", BRONZE_CALLS)
print("Silver calls :", SILVER_CALLS)
print("Silver rejects:", SILVER_REJECTS)

Bronze calls : workspace.capstone_tanmaya.bronze_calls
Silver calls : workspace.capstone_tanmaya.silver_calls
Silver rejects: workspace.capstone_tanmaya.silver_rejects


In [0]:
# ============================================================
# READ BRONZE CALLS
# ============================================================

bronze = spark.table(BRONZE_CALLS)

print("Bronze rows:", bronze.count())

Bronze rows: 376200


In [0]:
# ============================================================
# READ TOWER MASTER
# ============================================================

BRONZE_TOWERS = f"{CATALOG}.{SCHEMA}.bronze_towers"

towers = spark.table(BRONZE_TOWERS)

print("Tower master rows:", towers.count())

Tower master rows: 120


In [0]:
# ============================================================
# SAFE TYPE CONVERSION + NORMALIZATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_prepared = (
    bronze
    .withColumn(
        "duration_int",
        F.expr("try_cast(duration_seconds AS INT)")
    )
    .withColumn(
        "start_timestamp",
        F.expr("try_cast(start_ts AS TIMESTAMP)")
    )
    .withColumn(
        "normalized_end_cause",
        F.upper(F.trim(F.col("end_cause")))
    )
)

print("Preparation complete.")

Preparation complete.


In [0]:
# ============================================================
# DEDUPLICATE ON CALL_ID
# ============================================================

dedupe_window = (
    Window
    .partitionBy("call_id")
    .orderBy(F.col("_ingested_at").desc())
)

deduped = (
    silver_prepared
    .withColumn(
        "_dedupe_rank",
        F.row_number().over(dedupe_window)
    )
    .filter(F.col("_dedupe_rank") == 1)
    .drop("_dedupe_rank")
)

bronze_count = bronze.count()
deduped_count = deduped.count()
duplicates_removed = bronze_count - deduped_count

print("Bronze rows       :", bronze_count)
print("After deduplication:", deduped_count)
print("Duplicates removed :", duplicates_removed)

Bronze rows       : 376200
After deduplication: 374400
Duplicates removed : 1800


In [0]:
# ============================================================
# ASSIGN REJECTION REASONS
# ============================================================

tower_ids = towers.select("tower_id").dropDuplicates()

validated = (
    deduped
    .join(
        tower_ids.withColumn("_tower_exists", F.lit(True)),
        on="tower_id",
        how="left"
    )
    .withColumn(
        "reject_reason",
        F.when(
            F.col("duration_int").isNull(),
            F.lit("duration_not_numeric")
        )
        .when(
            F.col("_tower_exists").isNull(),
            F.lit("unknown_tower")
        )
        .when(
            (F.col("duration_int") < 0) |
            (F.col("duration_int") > 7200),
            F.lit("duration_out_of_range")
        )
    )
    .drop("_tower_exists")
)

In [0]:
# ============================================================
# SILVER REJECTS
# ============================================================

silver_rejects = (
    validated
    .filter(F.col("reject_reason").isNotNull())
    .select(
        "call_id",
        "subscriber_id",
        "tower_id",
        "start_ts",
        "duration_seconds",
        "end_cause",
        "direction",
        "technology",
        "reject_reason",
        "_source_file",
        "_ingested_at",
        "_row_hash"
    )
)

print("Rejected rows:", silver_rejects.count())

Rejected rows: 2250


In [0]:
# ============================================================
# SAVE SILVER REJECTS
# ============================================================

silver_rejects.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_REJECTS)

print("silver_rejects saved successfully.")

silver_rejects saved successfully.


In [0]:
# ============================================================
# CREATE VALID SILVER CALLS
# ============================================================

silver_valid = (
    validated
    .filter(F.col("reject_reason").isNull())
    .drop("reject_reason")
    .withColumnRenamed(
        "duration_int",
        "duration_seconds_int"
    )
    .withColumnRenamed(
        "start_timestamp",
        "start_ts_timestamp"
    )
)

print("Valid Silver rows:", silver_valid.count())

Valid Silver rows: 372150


In [0]:
# ============================================================
# DERIVE BUSINESS FIELDS
# ============================================================

silver_valid = (
    silver_valid
    .withColumn(
        "is_dropped",
        F.when(
            F.col("normalized_end_cause") == "DROPPED",
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "end_ts",
        F.expr(
            "start_ts_timestamp + make_interval(0, 0, 0, 0, 0, 0, duration_seconds_int)"
        )
    )
)

In [0]:
# ============================================================
# ADD TOWER MASTER ATTRIBUTES
# ============================================================

tower_attributes = towers.select(
    "tower_id",
    "site_name",
    "city",
    "district",
    "capacity_channels",
    "commissioned_on"
)

silver_calls_final = (
    silver_valid
    .join(
        tower_attributes,
        on="tower_id",
        how="left"
    )
)

print("Final Silver rows:", silver_calls_final.count())

Final Silver rows: 372150


In [0]:
# ============================================================
# SAVE SILVER CALLS
# ============================================================

silver_calls_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_CALLS)

print("silver_calls saved successfully.")

silver_calls saved successfully.


In [0]:
# ============================================================
# SILVER VALIDATION
# ============================================================

silver_count = spark.table(SILVER_CALLS).count()
reject_count = spark.table(SILVER_REJECTS).count()

print("============================================")
print("TOPIC 31 — SILVER VALIDATION")
print("============================================")
print("Bronze calls       :", bronze_count)
print("Duplicates removed :", duplicates_removed)
print("Silver calls       :", silver_count)
print("Rejected rows      :", reject_count)
print("============================================")

assert bronze_count == 376200
assert duplicates_removed == 1800
assert silver_count == 372150

print("Silver count validation PASSED.")

TOPIC 31 — SILVER VALIDATION
Bronze calls       : 376200
Duplicates removed : 1800
Silver calls       : 372150
Rejected rows      : 2250
Silver count validation PASSED.


In [0]:
# ============================================================
# REJECTION REASON VALIDATION
# ============================================================

spark.sql(f"""
SELECT
    reject_reason,
    COUNT(*) AS rejected_rows
FROM {SILVER_REJECTS}
GROUP BY reject_reason
ORDER BY reject_reason
""").show()

+--------------------+-------------+
|       reject_reason|rejected_rows|
+--------------------+-------------+
|duration_not_numeric|         1200|
|duration_out_of_r...|          450|
|       unknown_tower|          600|
+--------------------+-------------+



In [0]:
# ============================================================
# FINAL SILVER SUMMARY
# ============================================================

print("============================================")
print("TOPIC 31 — SILVER COMPLETE")
print("============================================")
print("Bronze rows       :", bronze_count)
print("Duplicates removed:", duplicates_removed)
print("Silver valid rows :", silver_count)
print("Rejected rows     :", reject_count)
print("============================================")

print("Expected:")
print("Bronze            = 376200")
print("Duplicates removed= 1800")
print("Silver            = 372150")
print("============================================")

TOPIC 31 — SILVER COMPLETE
Bronze rows       : 376200
Duplicates removed: 1800
Silver valid rows : 372150
Rejected rows     : 2250
Expected:
Bronze            = 376200
Duplicates removed= 1800
Silver            = 372150
